<a href="https://colab.research.google.com/github/yun080926/Titanic/blob/main/titanic%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 코렙 한글깨짐 방지
import matplotlib.font_manager as fm
import matplotlib as mpl # Import matplotlib directly
import matplotlib.pyplot as plt # Import matplotlib.pyplot

!apt-get update -qq
!apt-get install fonts-nanum* -qq
sys_font=fm.findSystemFonts()
nanum_font = [f for f in sys_font if 'Nanum' in f]
nanum_font
fe = fm.FontEntry(
    fname=r'/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 데이터 불러오기
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

test_ids = test['PassengerId']  # 나중에 제출용으로 저장

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Selecting previously unselected package fonts-nanum-coding.
Preparing to unpack .../fonts-nanum-coding_2.5-3_all.deb ...
Unpacking fonts-nanum-coding (2.5-3) ...
Selecting previously unselected package fonts-nanum-eco.
Preparing to unpack .../fonts-nanum-eco_1.000-7_all.deb ...
Unpacking fonts-nanum-eco (1.000-7) ...
Selecting previously unselected package fonts-nanum-extra.
Preparing to unpack .../fonts-nanum-extra_20200506-1_all.deb ...
Unpacking fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Setting up fo

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
# 1️⃣ 결측치 처리
# 결측치 개수 확인
print("=== train 결측치 ===")
print(train.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 결측치를 히트맵으로 시각화
plt.figure(figsize=(10, 4))
sns.heatmap(train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('결측치 시각화 (노란색 = 결측치)')
plt.show()

In [ ]:
# 1. Cabin → 삭제
train = train.drop(columns=['Cabin'])

# 2. Age → 중앙값으로 채우기
print("Age 중앙값:", train['Age'].median())  # 얼마인지 확인
train['Age'] = train['Age'].fillna(train['Age'].median())

# 3. Embarked → 최빈값으로 채우기
print("Embarked 최빈값:", train['Embarked'].mode()[0])  # 얼마인지 확인
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# 처리 후 결측치 확인
print("\n=== 처리 후 결측치 ===")
print(train.isnull().sum())

결측치 처리 설명
- Cabin: 삭제 (결측치 77%, 채워도 의미없음)
- Age: 중앙값(28) (극단값 영향 적은 중앙값이 적합)
- Embarked: 최빈값(S) (글자라 평균 못씀, 가장 많은 값으로 대체)

In [ ]:
# 2️⃣ 데이터 인코딩
# 먼저 현재 컬럼 타입 확인
print(train.dtypes)

컬럼 타입 해석
- int64, float64 = 숫자 → 이미 OK
- object = 글자 → 인코딩 필요

In [ ]:
# Sex 인코딩 → Label Encoding (순서가 없지만 컬럼 2개뿐이라)
# male → 0, female → 1
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})

print(train['Sex'].value_counts())

Label Encoding 선택 이유

: male/female은 딱 2개뿐임. 2개짜리는 0/1로 표현 가능하므로

One-Hot 대신 Label Encoding 사용

In [ ]:
# Embarked 인코딩 → One-Hot Encoding (순서가 없는 경우)
# S, C, Q → 각각 0/1 컬럼으로 분리
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True) #drop_first=True다중공선성 방지

print(train[['Embarked_Q', 'Embarked_S']].head(10))
print("\n새로운 컬럼들:", train.columns.tolist())

In [ ]:
# True/False → 1/0 으로 변환
train['Embarked_Q'] = train['Embarked_Q'].astype(int)
train['Embarked_S'] = train['Embarked_S'].astype(int)

print(train[['Embarked_Q', 'Embarked_S']].head(10))

One-Hot Encoding 선택 이유

: S, C, Q는 그냥 항구 이름이라 순서가 없음. 각각 별개의 컬럼으로 생성

drop_first=True

S/C/Q 3개를 컬럼으로 만들면 2개만 있어도 나머지 하나를 알 수 있음.

그래서 첫 번째(C) 컬럼은 삭제 (다중공선성 방지)

In [ ]:
# 3️⃣ 데이터 스케일링
print(train[['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']].describe())

결과 해석
- Fare가 문제: 평균이 32인데 최대값이 512 (평균의 16배) → 이상치 존재

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

# Age, Fare만 스케일링 (범위 차이가 큰 컬럼들)
cols_to_scale = ['Age', 'Fare']
train[cols_to_scale] = scaler.fit_transform(train[cols_to_scale])

print(train[['Age', 'Fare']].describe())

3번 스케일링 완료

- 스케일링 대상: Age, Fare
- 선택한 방법: RobustScaler
- 이유: Fare에 이상치(512) 존재, 이상치에 강한 Robust 선택
- RandomForest엔 필수는 아니지만 다른 모델 대비 해둠

In [ ]:
# 4️⃣ 데이터 왜도 (Skewness)
# 각 컬럼 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train['Age'].hist(ax=axes[0], bins=30, color='skyblue')
axes[0].set_title('Age 분포')

train['Fare'].hist(ax=axes[1], bins=30, color='salmon')
axes[1].set_title('Fare 분포')

train['Pclass'].hist(ax=axes[2], bins=10, color='lightgreen')
axes[2].set_title('Pclass 분포')

plt.tight_layout()
plt.show()

In [ ]:
# 왜도 수치 계산
print("왜도 수치:")
print(f"Age  : {train['Age'].skew():.4f}")
print(f"Fare : {train['Fare'].skew():.4f}")
print(f"SibSp: {train['SibSp'].skew():.4f}")
print(f"Parch: {train['Parch'].skew():.4f}")

In [ ]:
# 로그 변환 (0이 있으면 log(0)= -무한대라 +1 해줌)
train['Fare'] = np.log1p(train['Fare'])
train['SibSp'] = np.log1p(train['SibSp'])
train['Parch'] = np.log1p(train['Parch'])

# 변환 후 왜도 확인
print("변환 후 왜도:")
print(f"Fare : {train['Fare'].skew():.4f}")
print(f"SibSp: {train['SibSp'].skew():.4f}")
print(f"Parch: {train['Parch'].skew():.4f}")


In [ ]:
# 시각화로 전후 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train['Fare'].hist(ax=axes[0], bins=30, color='salmon')
axes[0].set_title('Fare (log 변환 후)')

train['SibSp'].hist(ax=axes[1], bins=30, color='skyblue')
axes[1].set_title('SibSp (log 변환 후)')

plt.tight_layout()
plt.show()

그래프 해석

Fare (빨간색)
- 변환 전: 0 근처에 500개 몰리고 오른쪽으로 꼬리가 21까지
- 변환 후: 훨씬 고르게 퍼짐

SibSp (파란색):
- 0인 사람이 600명으로 압도적
- → 혼자 탑승한 사람이 그만큼 많다는 뜻
- → 데이터 특성상 한계

--

4번 왜도 처리 완료

- 왜도 심한 컬럼: Fare(4.79), SibSp(3.70), Parch(2.75)
- 처리 방법: 로그 변환 (log1p)
- 결과: 왜도 수치 절반 이상 감소
- 한계SibSp, Parch는 0이 너무 많아 완전 해소 어려움

In [ ]:
# 5️⃣ 이상치 (Outliers)
# 먼저 박스플롯으로 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train.boxplot(column='Age', ax=axes[0])
axes[0].set_title('Age')

train.boxplot(column='Fare', ax=axes[1])
axes[1].set_title('Fare')

train.boxplot(column='SibSp', ax=axes[2])
axes[2].set_title('SibSp')

plt.tight_layout()
plt.show()

In [ ]:
# IQR 방법으로 이상치 개수 확인
for col in ['Age', 'Fare', 'SibSp']:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = train[(train[col] < lower) | (train[col] > upper)]
    print(f"{col}: 이상치 {len(outliers)}개 (기준: {lower:.2f} ~ {upper:.2f})")

### 5. 이상치 처리

**이상치 기준:** IQR x 1.5 방법 사용
- Age: 66개, Fare: 31개, SibSp: 12개 탐지

**처리 결정: 제거 안 함**
- 이유 1: 탐지된 값들이 실제로 가능한 값
- 이유 2: RandomForest는 트리 기반이라 이상치 영향 적음
- 이유 3: 데이터가 891개로 많지 않아 제거 시 정보 손실 우려

In [ ]:
# 6️⃣ 피처 선택 및 생성
# 먼저 현재 컬럼 확인
print(train.columns.tolist())
print(train.shape)

In [ ]:

# 가족 수
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize']  = test['SibSp']  + test['Parch']  + 1

# 혼자 탑승 여부
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone']  = (test['FamilySize']  == 1).astype(int)

# 호칭 추출
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title']  = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare = ['Dr','Rev','Col','Major','Countess','Sir','Jonkheer','Lady','Capt','Don']
for df in [train, test]:
    df['Title'] = df['Title'].replace(rare, 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# 불필요한 컬럼 제거
train = train.drop(columns=['PassengerId', 'Name', 'Ticket'])
test  = test.drop(columns=['PassengerId', 'Name', 'Ticket'])

# Title 인코딩
train = pd.get_dummies(train, columns=['Title'], drop_first=True, dtype=int)
test  = pd.get_dummies(test,  columns=['Title'], drop_first=True, dtype=int)

print(train.columns.tolist())
print(train.shape)

In [ ]:
# 다중공선성 확인
# 상관관계 히트맵
plt.figure(figsize=(12, 8))
sns.heatmap(train.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation')
plt.show()

히트맵 해석
색깔 기준:
- 빨강 (1.0에 가까움)  → 강한 양의 상관관계
- 파랑 (-1.0에 가까움) → 강한 음의 상관관계
- 흰색 (0에 가까움)    → 거의 관계 없음

다중공선성 의심 컬럼들
1. SibSp & FamilySize → 0.88
2. Parch & FamilySize → 0.84
3. FamilySize & IsAlone → -0.82
4. Sex & Title_Mr → -0.87

In [ ]:
# SibSp, Parch → FamilySize로 합쳤으니 제거
# IsAlone → FamilySize랑 겹치니 제거
# Sex → Title로 대체 가능하니 제거

train = train.drop(columns=['SibSp', 'Parch', 'IsAlone', 'Sex'])
test  = test.drop(columns=['SibSp', 'Parch', 'IsAlone', 'Sex'])

print(train.columns.tolist())

새로 만든 피처:
- FamilySize: /SibSp + Parch + 1/ 가족 규모가 생존에 영향
- IsAlone: /FamilySize == 1/ 혼자 탄 사람이 더 위험
- Title_Mr/Mrs/Miss/Rare: /이름에서 호칭 추출/ 성별/나이/신분 함축

다중공선성 처리:

제거한 컬럼
- SibSp, Parch: FamilySize에 이미 포함 (상관관계 0.88, 0.84)
- IsAlone: FamilySize와 반대 방향으로 같은 정보 (-0.82)
- Sex: Title_Mr과 거의 같은 정보 (-0.87)

In [ ]:
#생존/사망 비율 확인
print("생존/사망 비율:")
print(train['Survived'].value_counts())
print()
print(f"사망: {train['Survived'].value_counts()[0]/len(train)*100:.1f}%")
print(f"생존: {train['Survived'].value_counts()[1]/len(train)*100:.1f}%")

결과 해석
- 사망: 61.6%
- 생존: 38.4%
- → 비율이 꽤 차이남 (23.2%)

비율이 크게 다르므로 K-Fold (Fold에 따라 생존자 차이가 클 수 있음) 대신

→ Stratified K-Fold 사용 (각 Fold마다 61.6% / 38.4% 비율을 유지)

In [ ]:
# Stratified K-Fold -> 각 Fold마다 61.6% / 38.4% 비율을 유지
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# 입력/정답 분리
X = train.drop(columns=['Survived'])
y = train['Survived']

# Stratified K-Fold 설정
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 모델
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 검증
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("각 Fold 정확도:", scores.round(4))
print(f"평균 정확도: {scores.mean():.4f}")
print(f"표준편차: {scores.std():.4f}")

결과 해석
- 1번 Fold: 83.2%
- 2번 Fold: 79.8%
- 3번 Fold: 80.3%
- 4번 Fold: 83.7%
- 5번 Fold: 82.6%
- 평균: 81.9%
- 표준편차 0.0158 → 5번 모두 비슷한 점수,
Fold마다 크게 차이 안 나므로 모델이 안정적

In [ ]:
# 8️⃣ 모델 선택
# 피처 중요도 확인(어떤 피처가 생존에 가장 영향을 줬는지)

# 일단 모델 학습
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# 피처 중요도 시각화
importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance')
plt.show()

생존 여부 → 0 또는 1
→ "분류(Classification)" 문제

분류 문제에 쓸 수 있는 모델 중 Random Forest
1. 생존 여부는 선형 관계가 아님
   (나이가 많을수록 생존률이 낮다 같은
   단순한 관계가 아님)

2. 여러 트리의 다수결 → 과적합 방지

3. 이상치에 강함 (데이터에 이상치가 있었음)

4. 피처 중요도를 알 수 있음
   → 어떤 컬럼이 생존에 영향을 줬는지 확인 가능

--

피처 중요도 해석
- 1위. Fare     (25%) → 요금이 생존에 가장 큰 영향
- 2위. Age      (23%) → 나이
- 3위. Title_Mr (20%) → 남성 여부
- 4위. Pclass   (8%)  → 좌석 등급
- 5위. FamilySize(8%) → 가족 규모
...
- Embarked_Q    (1%)  → 탑승 항구는 거의 영향 없음

In [ ]:
# 하이퍼파라미터 튜닝
# Grid Search = 모든 조합을 다 해봐서 최적값 찾기
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X, y)

print("최적 파라미터:", grid_search.best_params_)
print(f"최적 정확도: {grid_search.best_score_:.4f}")

RandomForest 예시:
- n_estimators → 트리 몇 개 만들지 (기본값: 100)
- max_depth    → 트리를 얼마나 깊게 만들지
- min_samples_split → 몇 개 이상일 때 가지를 나눌지

Grid Search = 모든 조합을 다 해봐서 최적값 찾기
- n_estimators: 100, 200, 300
- max_depth: 3, 5, 7
- → 3 x 3 = 9가지 조합을 전부 시도

--

결과 해석

튜닝 전후 비교:
- 튜닝 전: 81.9%
- 튜닝 후: 84.2% → 2.3% 상승

최적 파라미터 의미:
- n_estimators=100   → 트리 100개 (기본값으로 충분했음)
- max_depth=7        → 트리 깊이 7단계까지만
                     (너무 깊으면 과적합!)
- min_samples_split=10 → 10개 이상일 때만 가지 나누기
                      (너무 잘게 나누면 과적합!)

In [ ]:
# 대안 모델 비교
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

models = {
    'RandomForest': RandomForestClassifier(max_depth=7, min_samples_split=10, n_estimators=100, random_state=42),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

for name, m in models.items():
    score = cross_val_score(m, X, y, cv=skf, scoring='accuracy')
    print(f"{name}: {score.mean():.4f} (+/- {score.std():.4f})")

모델 비교 결과
- 1위. RandomForest    84.2% ← 선택한 모델
- 2위. LogisticRegression 82.4%
- 3위. SVM             82.0%
- 4위. XGBoost         81.1%

In [ ]:
# 9️⃣ 모델 평가
# Confusion Matrix 시각화
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 최적 모델로 예측
best_model = RandomForestClassifier(
    max_depth=7,
    min_samples_split=10,
    n_estimators=100,
    random_state=42
)

# 교차검증으로 예측값 뽑기
y_pred = cross_val_predict(best_model, X, y, cv=skf)

# Confusion Matrix 시각화
cm = confusion_matrix(y, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['사망(0)', '생존(1)'],
            yticklabels=['사망(0)', '생존(1)'])
plt.title('Confusion Matrix')
plt.ylabel('실제값')
plt.xlabel('예측값')
plt.show()

결과 해석
- 503 → 실제 사망, 예측도 사망 ✅ (잘 맞춤)
- 247 → 실제 생존, 예측도 생존 ✅ (잘 맞춤)
- 46  → 실제 사망인데 생존으로 예측 ❌ (오답)
- 95  → 실제 생존인데 사망으로 예측 ❌ (오답)

생존자(342명) 중 95명을 사망으로 잘못 예측
→ 실제 생존자를 놓치는 경우

In [ ]:
# 수치 지표 확인
print(classification_report(y, y_pred, target_names=['사망(0)', '생존(1)']))

결과 해석

사망(0) 예측:
- Precision 0.84 → 사망이라 예측한 것 중 84%가 실제 사망
- Recall    0.92 → 실제 사망자 중 92%를 잡아냄 (잘함!)
- F1-score  0.88 → 균형 잡힌 점수

생존(1) 예측:
= Precision 0.84 → 생존이라 예측한 것 중 84%가 실제 생존
- Recall    0.72 → 실제 생존자 중 72%만 잡아냄 (아쉬움!)
- F1-score  0.78 → 사망보다 낮음

생존자 Recall이 72%
→ 실제 생존자 342명 중 95명을
  "사망"으로 잘못 예측하고 있음

F1-score를 주요 지표로 선택한 이유:

Accuracy만 보면 부족

생존/사망 비율이 62:38로 불균형하기 때문에
Precision과 Recall을 동시에 고려하는
F1-score가 더 적합

In [ ]:
# 🔟 과적합 방지
# 모델 과적합 확인
# train 정확도 vs validation 정확도 비교
best_model = RandomForestClassifier(
    max_depth=7,
    min_samples_split=10,
    n_estimators=100,
    random_state=42
)

best_model.fit(X, y)
train_score = best_model.score(X, y)
val_score = cross_val_score(best_model, X, y, cv=skf, scoring='accuracy').mean()

print(f"Train 정확도: {train_score:.4f}")
print(f"Validation 정확도: {val_score:.4f}")
print(f"차이: {train_score - val_score:.4f}")

결과 해석
- Train 정확도:      87.5%
- Validation 정확도: 84.2%
- 차이:               3.3% (차이가 3.3%로 과적합 없음)

과적합 판단 기준
- 차이 5% 이하  → 괜찮음
- 차이 5~10%   → 약간 과적합 주의
- 차이 10% 이상 → 과적합

과적합이 없는 이유
- max_depth=7        → 트리를 너무 깊게 안 만듦
- min_samples_split=10 → 너무 잘게 안 나눔
- → 이 두 개가 이미 과적합 방지 역할

In [ ]:
from sklearn.linear_model import LogisticRegression

# 규제 없음
lr_none = LogisticRegression(penalty=None, max_iter=1000, random_state=42)
# L1 규제
lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', random_state=42)
# L2 규제
lr_l2 = LogisticRegression(penalty='l2', max_iter=1000, random_state=42)

models = {
    'LR 규제없음': lr_none,
    'LR L1 규제': lr_l1,
    'LR L2 규제': lr_l2
}

for name, m in models.items():
    m.fit(X, y)
    train_acc = m.score(X, y)
    val_acc = cross_val_score(m, X, y, cv=skf, scoring='accuracy').mean()
    print(f"{name} → Train: {train_acc:.4f}, Val: {val_acc:.4f}, 차이: {train_acc-val_acc:.4f}")

RandomForest는 L1/L2 규제가 없어서 Logistic Regression으로 비교
결과 해석
- LR 규제없음 → 차이 0.0056 (0.56%) (Train이 Val보다 0.56% 높음)
- LR L1 규제 → 차이 0.0011 (0.11%) (Train이 Val보다 0.11% 높음 (차이 줄었음))
- LR L2 규제 → 차이 -0.001 (거의 0%) (Val이 Train보다 오히려 높음 (균형))
